Hello, this is el mehdi bensouda

# GTEx Gene Expression Data Exploration

This notebook is used to explore the GTEx V11 median TPM gene expression dataset.

The goal is to understand the structure of the data, check the genes and tissues, and prepare the dataset for further analysis.

In [27]:
%pip install pandas numpy matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

The first two rows of the GCT file contain file information, so they need to be skipped.

In [29]:
file_name = "../data/GTEx_Analysis_2025-08-22_v11_RNASeQCv2.4.3_gene_median_tpm.gct.gz"

df = pd.read_csv(
    file_name,
    sep="\t",
    skiprows=2
)

df.head()

,Name,Description,Adipose_Subcutaneous,Adipose_Visceral_Omentum,Adrenal_Gland,Artery_Aorta,Artery_Coronary,Artery_Tibial,Bladder,Brain_Amygdala,...,Spleen,Stomach,Stomach_Mixed_Cell,Stomach_Mucosa,Stomach_Muscularis,Testis,Thyroid,Uterus,Vagina,Whole_Blood
0,ENSG00000290825.2,DDX11L16,0.00000,0.0000,0.00000,0.0000,0.00000,0.00000,0.00000,0.011780,...,0.00000,0.00000,0.014928,0.012652,0.015991,0.073208,0.00000,0.00000,0.00000,0.0000
1,ENSG00000223972.6,DDX11L1,0.00000,0.0000,0.00000,0.0000,0.00000,0.00000,0.00000,0.000000,...,0.00000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.00000,0.0000
2,ENSG00000310526.1,WASH7P,2.45431,1.9053,1.75253,2.3178,2.21944,1.66539,3.14825,0.565228,...,5.04503,2.04517,1.868420,1.507490,1.909350,3.126710,4.01324,4.92657,3.85789,2.2098
3,ENSG00000243485.6,MIR1302-2HG,0.00000,0.0000,0.00000,0.0000,0.00000,0.00000,0.00000,0.000000,...,0.00000,0.00000,0.000000,0.000000,0.000000,0.036257,0.00000,0.00000,0.00000,0.0000
4,ENSG00000237613.3,FAM138A,0.00000,0.0000,0.00000,0.0000,0.00000,0.00000,0.00000,0.012860,...,0.00000,0.00000,0.015325,0.013091,0.018168,0.000000,0.00000,0.00000,0.00000,0.0000


In [30]:
# check the size of the data
df.shape

(74628, 70)

In [31]:
# check the data types of the columns
df.dtypes

Name                            str
Description                     str
Adipose_Subcutaneous        float64
Adipose_Visceral_Omentum    float64
Adrenal_Gland               float64
                             ...   
Testis                      float64
Thyroid                     float64
Uterus                      float64
Vagina                      float64
Whole_Blood                 float64
Length: 70, dtype: object

In [32]:
# check the column names
df.columns

Index(['Name', 'Description', 'Adipose_Subcutaneous',
       'Adipose_Visceral_Omentum', 'Adrenal_Gland', 'Artery_Aorta',
       'Artery_Coronary', 'Artery_Tibial', 'Bladder', 'Brain_Amygdala',
       'Brain_Anterior_cingulate_cortex_BA24', 'Brain_Caudate_basal_ganglia',
       'Brain_Cerebellar_Hemisphere', 'Brain_Cerebellum', 'Brain_Cortex',
       'Brain_Frontal_Cortex_BA9', 'Brain_Hippocampus', 'Brain_Hypothalamus',
       'Brain_Nucleus_accumbens_basal_ganglia', 'Brain_Putamen_basal_ganglia',
       'Brain_Spinal_cord_cervical_c-1', 'Brain_Substantia_nigra',
       'Breast_Mammary_Tissue', 'Cells_Cultured_fibroblasts',
       'Cells_EBV-transformed_lymphocytes', 'Cervix_Ectocervix',
       'Cervix_Endocervix', 'Colon_Sigmoid', 'Colon_Transverse',
       'Colon_Transverse_Mixed_Cell', 'Colon_Transverse_Mucosa',
       'Colon_Transverse_Muscularis', 'Esophagus_Gastroesophageal_Junction',
       'Esophagus_Mucosa', 'Esophagus_Muscularis', 'Fallopian_Tube',
       'Heart_Atrial_Append

In [33]:
#check for missing values
df.isnull().sum()

Name                        0
Description                 0
Adipose_Subcutaneous        0
Adipose_Visceral_Omentum    0
Adrenal_Gland               0
                           ..
Testis                      0
Thyroid                     0
Uterus                      0
Vagina                      0
Whole_Blood                 0
Length: 70, dtype: int64

In [34]:
#check yne gene names
df[["Name", "Description"]].head(20)

,Name,Description
0,ENSG00000290825.2,DDX11L16
1,ENSG00000223972.6,DDX11L1
2,ENSG00000310526.1,WASH7P
3,ENSG00000243485.6,MIR1302-2HG
4,ENSG00000237613.3,FAM138A
5,ENSG00000308361.1,ENSG00000308361
6,ENSG00000290826.2,ENSG00000290826
7,ENSG00000268020.3,OR4G4P
8,ENSG00000186092.7,OR4F5
9,ENSG00000241860.8,ENSG00000241860


## Create the Gene-Tissue Expression Table

I will create a table where each row represents a gene and each column represents a tissue. The values represent the median TPM expression of each gene in each tissue.

In [35]:
# get all the tissue columns
tissue_cols = df.columns[2:]

print("Number of tissues:", len(tissue_cols))

Number of tissues: 68


In [36]:
# create a table with the gene information and tissue expression
gene_tissue_table = df[
    ["Name", "Description"] + list(tissue_cols)
].copy()

In [37]:
#check if all the genes present and are all the tissue columns present
tissue_cols.tolist()

['Adipose_Subcutaneous',
 'Adipose_Visceral_Omentum',
 'Adrenal_Gland',
 'Artery_Aorta',
 'Artery_Coronary',
 'Artery_Tibial',
 'Bladder',
 'Brain_Amygdala',
 'Brain_Anterior_cingulate_cortex_BA24',
 'Brain_Caudate_basal_ganglia',
 'Brain_Cerebellar_Hemisphere',
 'Brain_Cerebellum',
 'Brain_Cortex',
 'Brain_Frontal_Cortex_BA9',
 'Brain_Hippocampus',
 'Brain_Hypothalamus',
 'Brain_Nucleus_accumbens_basal_ganglia',
 'Brain_Putamen_basal_ganglia',
 'Brain_Spinal_cord_cervical_c-1',
 'Brain_Substantia_nigra',
 'Breast_Mammary_Tissue',
 'Cells_Cultured_fibroblasts',
 'Cells_EBV-transformed_lymphocytes',
 'Cervix_Ectocervix',
 'Cervix_Endocervix',
 'Colon_Sigmoid',
 'Colon_Transverse',
 'Colon_Transverse_Mixed_Cell',
 'Colon_Transverse_Mucosa',
 'Colon_Transverse_Muscularis',
 'Esophagus_Gastroesophageal_Junction',
 'Esophagus_Mucosa',
 'Esophagus_Muscularis',
 'Fallopian_Tube',
 'Heart_Atrial_Appendage',
 'Heart_Left_Ventricle',
 'Kidney_Cortex',
 'Kidney_Medulla',
 'Liver',
 'Liver_Hepatoc

In [38]:
# check how many tissue columns we have
len(tissue_cols)

68

In [39]:
# chech the data types
gene_tissue_table.dtypes

Name                            str
Description                     str
Adipose_Subcutaneous        float64
Adipose_Visceral_Omentum    float64
Adrenal_Gland               float64
                             ...   
Testis                      float64
Thyroid                     float64
Uterus                      float64
Vagina                      float64
Whole_Blood                 float64
Length: 70, dtype: object

In [40]:
gene_tissue_table.isnull().sum()

Name                        0
Description                 0
Adipose_Subcutaneous        0
Adipose_Visceral_Omentum    0
Adrenal_Gland               0
                           ..
Testis                      0
Thyroid                     0
Uterus                      0
Vagina                      0
Whole_Blood                 0
Length: 70, dtype: int64

In [41]:
# check wheteher some genes are missing in every tissue

# keep only the tissue expression values
expression_table = gene_tissue_table[tissue_cols]

# find genes that have 0 TPM in every tissue
zero_genes = (expression_table.sum(axis=1) == 0)

# count how many genes have 0 expression everywhere
zero_genes.sum()

np.int64(13875)

In [42]:
# count genes that are not zero in every tissue
(~zero_genes).sum()

np.int64(60753)

We remove all-zero genes because they cannot be normalized and they do not help us understand which tissues a gene is associated with.

In [43]:
gene_tissue_table.drop(
    gene_tissue_table[zero_genes].index,
    inplace=True)
# check the data size after dropping genes with 0 expression in every tissue
gene_tissue_table.shape

(60753, 70)

## Normalize Gene Expression
For each gene, I will convert the TPM values into relative expression values so that the values across all tissues add up to 1.

In [44]:
# get only the tissue values
expression_table = gene_tissue_table[tissue_cols]

# get the total expression for each gene
gene_total = expression_table.sum(axis=1)

# divide each tissue value by the total for that gene
normaliz_expression = expression_table.div(gene_total, axis=0)
normaliz_expression.head()

,Adipose_Subcutaneous,Adipose_Visceral_Omentum,Adrenal_Gland,Artery_Aorta,Artery_Coronary,Artery_Tibial,Bladder,Brain_Amygdala,Brain_Anterior_cingulate_cortex_BA24,Brain_Caudate_basal_ganglia,...,Spleen,Stomach,Stomach_Mixed_Cell,Stomach_Mucosa,Stomach_Muscularis,Testis,Thyroid,Uterus,Vagina,Whole_Blood
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.021687,0.000000,0.019531,...,0.000000,0.000000,0.027482,0.023292,0.029439,0.134775,0.000000,0.000000,0.000000,0.00000
2,0.014339,0.011131,0.010239,0.013541,0.012967,0.00973,0.018393,0.003302,0.003814,0.003990,...,0.029475,0.011948,0.010916,0.008807,0.011155,0.018267,0.023447,0.028782,0.022539,0.01291
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.266933,0.000000,0.000000,0.000000,0.00000
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.024759,0.025860,0.024015,...,0.000000,0.000000,0.029505,0.025204,0.034978,0.000000,0.000000,0.000000,0.000000,0.00000
5,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000


In [45]:
# check if each gene adds up to 1
normaliz_expression.sum(axis=1).head()

0    1.0
2    1.0
3    1.0
4    1.0
5    1.0
dtype: float64

In [46]:
# add the gene ID and gene name back to the normalized table
normaliz_gene_tissue_table = pd.concat(
    [
        gene_tissue_table[["Name", "Description"]].reset_index(drop=True),
        normaliz_expression.reset_index(drop=True)
    ],
    axis=1
)

normaliz_gene_tissue_table.head()

,Name,Description,Adipose_Subcutaneous,Adipose_Visceral_Omentum,Adrenal_Gland,Artery_Aorta,Artery_Coronary,Artery_Tibial,Bladder,Brain_Amygdala,...,Spleen,Stomach,Stomach_Mixed_Cell,Stomach_Mucosa,Stomach_Muscularis,Testis,Thyroid,Uterus,Vagina,Whole_Blood
0,ENSG00000290825.2,DDX11L16,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.021687,...,0.000000,0.000000,0.027482,0.023292,0.029439,0.134775,0.000000,0.000000,0.000000,0.00000
1,ENSG00000310526.1,WASH7P,0.014339,0.011131,0.010239,0.013541,0.012967,0.00973,0.018393,0.003302,...,0.029475,0.011948,0.010916,0.008807,0.011155,0.018267,0.023447,0.028782,0.022539,0.01291
2,ENSG00000243485.6,MIR1302-2HG,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.266933,0.000000,0.000000,0.000000,0.00000
3,ENSG00000237613.3,FAM138A,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.024759,...,0.000000,0.000000,0.029505,0.025204,0.034978,0.000000,0.000000,0.000000,0.000000,0.00000
4,ENSG00000308361.1,ENSG00000308361,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000


In [47]:
# check that each gene adds up to 1
normaliz_gene_tissue_table[tissue_cols].sum(axis=1).head()

0    1.0
1    1.0
2    1.0
3    1.0
4    1.0
dtype: float64

## Calculate Tissue Scores

In [48]:
# genes we want to test
input_genes = ["TP53", "BRCA1", "EGFR"]

input_genes

['TP53', 'BRCA1', 'EGFR']

In [49]:
# select the genes from the normalized table
selected_genes = normaliz_gene_tissue_table[
    normaliz_gene_tissue_table["Description"].isin(input_genes)
]

selected_genes

,Name,Description,Adipose_Subcutaneous,Adipose_Visceral_Omentum,Adrenal_Gland,Artery_Aorta,Artery_Coronary,Artery_Tibial,Bladder,Brain_Amygdala,...,Spleen,Stomach,Stomach_Mixed_Cell,Stomach_Mucosa,Stomach_Muscularis,Testis,Thyroid,Uterus,Vagina,Whole_Blood
23538,ENSG00000146648.21,EGFR,0.032606,0.021326,0.012016,0.019738,0.016661,0.027407,0.019391,0.004295,...,0.005528,0.008939,0.009711,0.004848,0.016575,0.004589,0.020933,0.018030,0.034686,0.000049
47942,ENSG00000141510.19,TP53,0.020197,0.016403,0.012350,0.017558,0.018793,0.018620,0.021942,0.003742,...,0.026948,0.011749,0.009746,0.009263,0.010963,0.014221,0.019135,0.028040,0.023253,0.010055
49233,ENSG00000012048.26,BRCA1,0.010521,0.008012,0.006617,0.008113,0.009167,0.007834,0.013337,0.006715,...,0.017476,0.008070,0.006860,0.004286,0.010779,0.076366,0.031782,0.016199,0.016319,0.008534


In [50]:
# add the normalized values for each tissue
tissue_scores = selected_genes[tissue_cols].sum(axis=0)

tissue_scores

Adipose_Subcutaneous        0.063324
Adipose_Visceral_Omentum    0.045741
Adrenal_Gland               0.030983
Artery_Aorta                0.045408
Artery_Coronary             0.044621
                              ...   
Testis                      0.095177
Thyroid                     0.071850
Uterus                      0.062268
Vagina                      0.074257
Whole_Blood                 0.018638
Length: 68, dtype: float64

In [51]:
# sort tissues from highest score to lowest
tissue_scores = tissue_scores.sort_values(ascending=False)

tissue_scores.head(10)

Cells_EBV-transformed_lymphocytes    0.223056
Cells_Cultured_fibroblasts           0.138872
Skin_Sun_Exposed_Lower_leg           0.110444
Skin_Not_Sun_Exposed_Suprapubic      0.108509
Testis                               0.095177
Nerve_Tibial                         0.076708
Vagina                               0.074257
Thyroid                              0.071850
Cervix_Ectocervix                    0.068521
Esophagus_Mucosa                     0.066168
dtype: float64

In [52]:
# make the tissue scores easier to read
tissue_score_table = tissue_scores.reset_index()

tissue_score_table.columns = ["Tissue", "Score"]

tissue_score_table.head(11)

,Tissue,Score
0,Cells_EBV-transformed_lymphocytes,0.223056
1,Cells_Cultured_fibroblasts,0.138872
2,Skin_Sun_Exposed_Lower_leg,0.110444
3,Skin_Not_Sun_Exposed_Suprapubic,0.108509
4,Testis,0.095177
5,Nerve_Tibial,0.076708
6,Vagina,0.074257
7,Thyroid,0.071850
8,Cervix_Ectocervix,0.068521
9,Esophagus_Mucosa,0.066168


## Random Gene Set Testing

We generate 1,000 random gene sets with the same number of genes as the real set and calculate their tissue scores. Then we compare the random scores with the real scores to see if the result could happen by chance.

In [70]:
# count how many genes are in the selected gene set
n_genes = len(selected_genes)

n_genes

3

In [71]:
# number of random tests
n_random = 1000

# store the random tissue scores
random_scores = []

In [72]:
for i in range(n_random):

    # select random genes with the same size as the real gene set
    random_genes = normaliz_gene_tissue_table.sample(
        n=n_genes,
        replace=False
    )

    # calculate tissue scores for the random genes
    random_tissue_scores = random_genes[tissue_cols].sum(axis=0)

    # save the scores
    random_scores.append(random_tissue_scores)

In [73]:
random_scores_table = pd.DataFrame(random_scores)

random_scores_table.head()

,Adipose_Subcutaneous,Adipose_Visceral_Omentum,Adrenal_Gland,Artery_Aorta,Artery_Coronary,Artery_Tibial,Bladder,Brain_Amygdala,Brain_Anterior_cingulate_cortex_BA24,Brain_Caudate_basal_ganglia,...,Spleen,Stomach,Stomach_Mixed_Cell,Stomach_Mucosa,Stomach_Muscularis,Testis,Thyroid,Uterus,Vagina,Whole_Blood
0,0.014436,0.012632,0.010707,0.016219,0.015218,0.015240,0.016720,0.011022,0.012464,0.009283,...,0.020881,0.011891,0.014926,0.012421,0.017600,2.016085,0.021084,0.018864,0.016698,0.006467
1,0.008471,0.008357,0.008596,0.016212,0.014484,0.010791,0.015099,0.028675,0.035876,0.043233,...,0.013688,0.010018,0.014919,0.008832,0.014656,1.065000,0.054488,0.020828,0.012586,0.009867
2,0.038284,0.030020,0.034828,0.131473,0.086187,0.094610,0.067674,0.008646,0.008276,0.009787,...,0.046077,0.021996,0.052577,0.015095,0.066697,0.098092,0.112177,0.077080,0.047537,0.010945
3,0.016052,0.012527,0.004054,0.004659,0.006303,0.004821,0.007102,0.004025,0.019505,0.004867,...,0.014673,0.010327,0.010753,0.011903,0.010832,0.438972,0.025883,0.006319,0.035801,0.013318
4,0.036269,0.023318,0.013654,0.015837,0.021315,0.014116,0.018224,0.004893,0.004446,0.007599,...,0.012269,0.010567,0.009181,0.004106,0.010112,1.010990,0.023262,0.025646,0.019276,0.000768


In [74]:
# get the highest tissue score from each random gene set
random_top_scores = random_scores_table.max(axis=1)

random_top_scores.head()

0    2.016085
1    1.065000
2    0.195359
3    1.023042
4    1.019903
dtype: float64

In [75]:
# calculate a p-value for each real tissue score
p_values = []

for tissue in tissue_cols:

    real_score = tissue_scores[tissue]

    p_value = (
        (random_top_scores >= real_score).sum() + 1
    ) / (n_random + 1)

    p_values.append(p_value)

In [76]:
tissue_score_table["P_value"] = p_values

In [77]:
# create a table with the tissue scores and p-values
p_value_table = pd.DataFrame({
    "Tissue": tissue_cols,
    "Score": [tissue_scores[tissue] for tissue in tissue_cols],
    "P_value": p_values
})

p_value_table = p_value_table.sort_values(
    "P_value"
)

p_value_table.head(10)

,Tissue,Score,P_value
22,Cells_EBV-transformed_lymphocytes,0.223056,0.918082
21,Cells_Cultured_fibroblasts,0.138872,0.977023
54,Skin_Sun_Exposed_Lower_leg,0.110444,0.988012
53,Skin_Not_Sun_Exposed_Suprapubic,0.108509,0.989011
63,Testis,0.095177,0.997003
4,Artery_Coronary,0.044621,1.000000
0,Adipose_Subcutaneous,0.063324,1.000000
1,Adipose_Visceral_Omentum,0.045741,1.000000
8,Brain_Anterior_cingulate_cortex_BA24,0.011822,1.000000
9,Brain_Caudate_basal_ganglia,0.017990,1.000000


We randomly selected gene sets with the same number of genes as our real gene set and repeated this 1,000 times. For each random gene set, we calculated the tissue scores using the same method as the real genes and saved the results in a table. We then used these random scores as a comparison to see whether the tissue scores from our real gene set are unusually high or could happen by chance.